In [0]:
import re
import pyspark.sql.functions as F
from delta import DeltaTable

# Data Cleansing

In [0]:
df = spark.read.table("fraud_detection_project.bronze_layer.security_logs")

def StandardizeNames(df):
    l = df.columns
    cols = []
    for c in l:
        temp = re.sub(r'(?<!^)(?=[A-Z])', '_', c).lower()
        for _ in range(5):
            temp = re.sub(r'\b([a-z])_([a-z])\b', r'\1\2', temp)
            temp = re.sub(r'(?<=[a-z])_([a-z])(?=_|$)', r'\1', temp)
        temp = re.sub(r'_+','_', temp)
        temp = temp.lstrip('_')
        cols.append(temp)
    return df.toDF(*cols)
df = StandardizeNames(df)
df.dtypes

In [0]:
# Deleting duplicated data
df.dropDuplicates(['security_logs_id'])

# Deleting rows without some features
df = df.dropna(how='any', subset=['security_logs_id','file_path','ingest_datetime'])

In [0]:
target = "fraud_detection_project.silver_layer.security_logs"

if spark.catalog.tableExists(target):
    dt = DeltaTable.forName(spark, target)

    dt.alias("t").merge(
        df.alias("s"),
        "t.security_logs_id = s.security_logs_id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
    print("Merge concluded.")
else:
    df.write.format("delta") \
      .option("mergeSchema", "true") \
      .saveAsTable(target)
    print("Tabel created.")

In [0]:
%sql
SELECT DISTINCT password_failures_session, count(password_failures_session) as quantity_times FROM fraud_detection_project.silver_layer.security_logs
GROUP BY password_failures_session